La hipótesis del trabajo es que las representaciones internas de un LLM se alinean más con las del cerebro humano leyendo utilizando un prompt que especifica cómo debe leer el texto. 

Este notebook extrae embeddings bajo múltiples condiciones de prompting, los alinea temporalmente con los volúmenes fMRI, y construye las matrices de diseño FIR para el encoding model. 

Además, Wehbe et al (2014) suma las features de las palabras por TR.

**Condiciones de prompting**

In [1]:
PROMPTS = {
    'neutral': "",  # Sin instrucción, base
    'literary': (
        "You are a literary critic. As you read this text, focus on "
        "narrative structure, character development, and prose style."
    ),
    'cognitive': (
        "You are a cognitive scientist. As you read, focus on the mental "
        "states, beliefs, desires, and intentions of each character."
    ),
    'emotional': (
        "You are reading this story with deep emotional engagement. "
        "Focus on the emotions felt by each character and the emotional "
        "tension of each scene."
    ),
    'syntactic': (
        "You are a linguist analyzing sentence structure. Focus on "
        "grammatical complexity, dependency relations, and word order."
    ),
}

In [ ]:
model_name = "gpt2" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, output_hidden_states=True, torch_dtype=torch.float16
)
model.eval()

def extract_embeddings(word_texts, system_prompt="", layer=-1):
    """
    Extrae un embedding por palabra, en contexto acumulativo.

    Para la palabra n, el input al modelo es:
    [system_prompt] + [palabra_1] + [palabra_2] + ... + [palabra_n]

    Se extrae el hidden state de la capa 'layer' en la última posición.
    """
    embeddings = []
    context = system_prompt

    for i, word in enumerate(word_texts):
        context += " " + word
        tokens = tokenizer(context, return_tensors="pt",
                          truncation=True, max_length=4096)

        with torch.no_grad():
            output = model(**tokens)

        # Embedding de la última posición, capa especificada
        hidden = output.hidden_states[layer][0, -1, :]
        embeddings.append(hidden.cpu().numpy())

        if (i + 1) % 500 == 0:
            print(f"  Palabra {i+1}/{len(word_texts)}")

    return np.array(embeddings)  # (n_words, embed_dim)

# Extraer para cada condición
for prompt_name, prompt_text in PROMPTS.items():
    print(f"\nExtrayendo embeddings: {prompt_name}")
    embeds = extract_embeddings(word_texts, system_prompt=prompt_text)
    np.save(f'embeddings_{prompt_name}.npy', embeds)
    print(f"  Shape: {embeds.shape}")